# LLooM Analysis: Inciting Speech False Negatives & False Positives

This notebook applies the LLooM pipeline to analyze misclassified samples from the INCITE dataset:

- **False Negatives (FN):** Gold = Inciting (Identity/Imputed Misdeeds/Exhortation), Predicted = None
- **False Positives (FP):** Gold = None, Predicted = Inciting (Identity/Imputed Misdeeds/Exhortation)

Pipeline: **Distill → Embed → Cluster → Synthesize Concepts → Score → Visualize**

Uses the OpenAI Chat Completions API. Set **`OPENAI_API_KEY`** before running. Models: **gpt-3.5-turbo** (distill + score), **gpt-5-mini** (synthesize concepts only).

In [ ]:
!pip install -q -U requests pandas scikit-learn sentence-transformers umap-learn hdbscan tqdm wordcloud squarify networkx


In [ ]:
import re, json, os, time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from wordcloud import WordCloud
import squarify

API_URL = "https://api.openai.com/v1/chat/completions"



# Load Inciting JSONL and extract FN / FP texts

In [ ]:
JSONL_PATH = "../Multi-Classification Run/inciting_gpt_5_mini_zero_shot.jsonl"  # adjust if running in Colab

OUTPUT_DIR = "outputs/lloom"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_DISTILL = "gpt-3.5-turbo"
MODEL_SYNTH = "gpt-5-mini"
MODEL_SCORE = "gpt-3.5-turbo"

INCITING_LABELS = {"Identity", "Imputed Misdeeds", "Exhortation"}

FN_SEED = "patterns in inciting speech that evades detection, including subtle identity attacks, veiled accusations of misdeeds, and indirect exhortations to act against target groups"
FP_SEED = "patterns in non-inciting content that gets falsely flagged as inciting, including factual reporting, counter-speech, and contextual misinterpretation of group references"

def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        content = f.read()
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\n\r":
            idx += 1
        if idx >= len(content):
            break
        obj, end_idx = decoder.raw_decode(content, idx)
        records.append(obj)
        idx = end_idx
    return records

def extract_reasoning(model_response: str) -> str:
    """Pull the reasoning section (section 3) from the structured model response."""
    match = re.search(
        r"3\.\s*Reasoning\s*:\s*(.+)",
        model_response,
        re.IGNORECASE | re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    return ""

records = load_jsonl(JSONL_PATH)
print(f"Total records: {len(records)}")

fn_texts, fp_texts = [], []
for rec in records:
    gold, pred = rec.get("gold_label", ""), rec.get("pred_label", "")
    reasoning = extract_reasoning(rec.get("model_response", ""))
    if not reasoning:
        continue
    if gold in INCITING_LABELS and pred == "None":
        fn_texts.append(reasoning)
    elif gold == "None" and pred in INCITING_LABELS:
        fp_texts.append(reasoning)

print(f"False Negatives (Inciting->None): {len(fn_texts)} reasoning texts")
print(f"False Positives (None->Inciting): {len(fp_texts)} reasoning texts")

# Helper functions

In [ ]:
def clean_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"http\S+|www\S+", " ", x)
    x = re.sub(r"&amp;|amp", " ", x)
    x = re.sub(r">+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def safe_json_load(text):
    text = str(text).strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        return json.loads(m.group(0))
    raise ValueError(f"Could not parse JSON from response:\n{text[:1200]}")

def _openai_chat_raw(prompt: str, model_name: str, temperature: float | None) -> str:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENAI_API_KEY (e.g. export OPENAI_API_KEY=sk-...)")
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "response_format": {"type": "json_object"},
    }
    if model_name.startswith("gpt-5") or model_name.startswith("o1") or model_name.startswith("o3"):
        payload["max_completion_tokens"] = 4096
        if model_name.startswith("gpt-5"):
            payload["seed"] = 42
    else:
        payload["max_tokens"] = 4096
        if temperature is not None:
            payload["temperature"] = temperature
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    if resp.status_code != 200:
        try:
            err = resp.json()
            msg = err.get("error", {}).get("message", str(err))
            typ = err.get("error", {}).get("type", "unknown")
            raise RuntimeError(f"API Error {resp.status_code} ({typ}): {msg}")
        except (ValueError, KeyError):
            raise RuntimeError(f"API Error {resp.status_code}: {resp.text[:500]}")
    return resp.json()["choices"][0]["message"]["content"]

def openai_chat_json(prompt, model_name, temperature=0.2):
    # gpt-5-mini and similar models do not support the temperature parameter
    if model_name.startswith("gpt-5") or model_name.startswith("o1") or model_name.startswith("o3"):
        temperature = None
    text = _openai_chat_raw(prompt, model_name, temperature)
    return safe_json_load(text)


# Step 1: Distill texts into bullets

In [ ]:
def distill_text_to_bullets(text, seed_phrase=""):
    prompt = f"""
You will read one social media text and summarize its main points into 2 to 4 short bullet phrases.

Rules:
- Each bullet must be 5 to 10 words
- Focus on the most salient meaning
- Avoid generic phrases
- Avoid repeating the same idea
- Keep wording concise
- Use the analytic focus only as a lens, not as a reason to invent content
- Return valid JSON only

Analytic focus:
{seed_phrase if seed_phrase else "None"}

Return format:
{{
  "bullets": ["...", "..."]
}}

TEXT:
{text}
"""
    obj = openai_chat_json(prompt, MODEL_DISTILL, temperature=0.2)
    bullets = obj.get("bullets", [])
    bullets = [str(b).strip() for b in bullets if str(b).strip()]
    return bullets[:4]

def _distill_one(idx, txt, seed_phrase="", max_retries=3, sleep_seconds=2):
    for attempt in range(max_retries):
        try:
            bullets = distill_text_to_bullets(txt, seed_phrase=seed_phrase)
            return {"orig_index": idx, "text": txt, "bullets": bullets, "error": None}
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(sleep_seconds * (attempt + 1))
            else:
                return {"orig_index": idx, "text": txt, "bullets": None, "error": str(e)}

def run_distillation(texts, seed_phrase="", max_workers=4, checkpoint_path="checkpoint.jsonl"):
    rows, done_ids = [], set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            for line in f:
                rec = json.loads(line)
                rows.append(rec)
                done_ids.add(rec["orig_index"])

    items = [(i, t) for i, t in enumerate(texts) if i not in done_ids]
    print(f"Already completed: {len(done_ids)}, Remaining: {len(items)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_distill_one, idx, txt, seed_phrase): idx
            for idx, txt in items
        }
        with open(checkpoint_path, "a") as fout:
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Distilling"):
                result = fut.result()
                fout.write(json.dumps(result) + "\n")
                fout.flush()
                rows.append(result)
    return pd.DataFrame(rows)

In [ ]:
fn_distilled_raw = run_distillation(
    fn_texts, seed_phrase=FN_SEED, max_workers=4,
    checkpoint_path=f"{OUTPUT_DIR}/fn_distill_checkpoint.jsonl"
)
fp_distilled_raw = run_distillation(
    fp_texts, seed_phrase=FP_SEED, max_workers=4,
    checkpoint_path=f"{OUTPUT_DIR}/fp_distill_checkpoint.jsonl"
)
print("FN distilled:", fn_distilled_raw.shape)
print("FP distilled:", fp_distilled_raw.shape)

# Step 2: Expand bullets, embed, and cluster

In [ ]:
def expand_distilled_rows(distilled_df):
    rows = []
    for _, row in distilled_df.iterrows():
        bullets = row.get("bullets")
        if isinstance(bullets, list):
            for b in bullets:
                rows.append({"orig_index": row["orig_index"], "text": row["text"], "bullet": b, "error": row.get("error")})
        else:
            rows.append({"orig_index": row["orig_index"], "text": row["text"], "bullet": None, "error": row.get("error")})
    return pd.DataFrame(rows)

fn_distilled = expand_distilled_rows(fn_distilled_raw).dropna(subset=["bullet"]).copy()
fp_distilled = expand_distilled_rows(fp_distilled_raw).dropna(subset=["bullet"]).copy()

print("Expanded FN bullets:", fn_distilled.shape)
print("Expanded FP bullets:", fp_distilled.shape)

fn_distilled.to_csv(f"{OUTPUT_DIR}/fn_distilled_bullets.csv", index=False)
fp_distilled.to_csv(f"{OUTPUT_DIR}/fp_distilled_bullets.csv", index=False)

In [ ]:
embedder = SentenceTransformer("all-mpnet-base-v2")

fn_emb = embedder.encode(fn_distilled["bullet"].tolist(), show_progress_bar=True)
fp_emb = embedder.encode(fp_distilled["bullet"].tolist(), show_progress_bar=True)

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
clusterer = HDBSCAN(min_cluster_size=12, metric="euclidean", cluster_selection_method="eom", prediction_data=True)

fn_red = umap_model.fit_transform(fn_emb)
fn_distilled["cluster"] = clusterer.fit_predict(fn_red)

fp_red = umap_model.fit_transform(fp_emb)
fp_distilled["cluster"] = clusterer.fit_predict(fp_red)

print("FN clusters:", fn_distilled["cluster"].value_counts().head(15))
print("\nFP clusters:", fp_distilled["cluster"].value_counts().head(15))

# Step 3: Synthesize concepts from clusters

In [ ]:
def synthesize_concept(cluster_bullets, seed_phrase=""):
    bullets_text = "\n".join(f"- {b}" for b in cluster_bullets)
    prompt = f"""
You are a qualitative-analysis assistant.
Below are bullet-point summaries drawn from a single cluster of social media texts.
Propose ONE high-level concept that unifies these bullets.

Rules:
- concept_name: 3-8 word descriptive label
- description: 1-2 sentences explaining the concept
- inclusion_criteria: concrete criteria for deciding if a new text matches
- example_signals: 3 short phrases a matching text might contain

Analytic focus:
{seed_phrase if seed_phrase else "None"}

Return format:
{{
  "concept_name": "...",
  "description": "...",
  "inclusion_criteria": "...",
  "example_signals": ["...", "...", "..."]
}}

BULLETS:
{bullets_text}
"""
    return openai_chat_json(prompt, MODEL_SYNTH)

def build_concepts(distilled_df, top_n=10, max_bullets=20, seed_phrase=""):
    concepts = []
    counts = distilled_df[distilled_df["cluster"] != -1]["cluster"].value_counts().head(top_n)
    for cid in tqdm(counts.index, desc="Synthesizing"):
        bullets = distilled_df[distilled_df["cluster"] == cid]["bullet"].dropna().tolist()[:max_bullets]
        try:
            c = synthesize_concept(bullets, seed_phrase=seed_phrase)
            c["cluster"], c["cluster_size"] = int(cid), int(counts[cid])
            concepts.append(c)
        except Exception as e:
            concepts.append({"cluster": int(cid), "cluster_size": int(counts[cid]),
                             "concept_name": None, "error": str(e)})
    return pd.DataFrame(concepts)

In [ ]:
fn_concepts = build_concepts(fn_distilled, top_n=10, seed_phrase=FN_SEED)
fp_concepts = build_concepts(fp_distilled, top_n=10, seed_phrase=FP_SEED)

display(fn_concepts)
display(fp_concepts)

fn_concepts.to_csv(f"{OUTPUT_DIR}/fn_concepts.csv", index=False)
fp_concepts.to_csv(f"{OUTPUT_DIR}/fp_concepts.csv", index=False)

# Step 4: Score every FN/FP text against concepts

In [ ]:
def score_text_against_concepts(text, concepts_df):
    valid = concepts_df.dropna(subset=["concept_name", "inclusion_criteria"])
    block = [f"Concept: {c['concept_name']}\nCriteria: {c['inclusion_criteria']}" for _, c in valid.iterrows()]
    prompt = f"""
You will read one social media text and decide which concepts apply.

Return valid JSON only in this format:
{{
  "matches": [
    {{"concept_name": "...", "match": 0, "reason": "short reason"}}
  ]
}}

Important:
- Include one entry for every concept provided
- Use match = 1 only if the text clearly fits the concept criteria
- Otherwise use match = 0

Concepts:
{chr(10).join(block)}

TEXT:
{text}
"""
    return openai_chat_json(prompt, MODEL_SCORE, temperature=0.1)["matches"]

def _score_one(idx, txt, concepts_df, max_retries=3, sleep_seconds=2):
    for attempt in range(max_retries):
        try:
            matches = score_text_against_concepts(txt, concepts_df)
            return {"orig_index": idx, "text": txt, "matches": matches, "error": None}
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(sleep_seconds * (attempt + 1))
            else:
                return {"orig_index": idx, "text": txt, "matches": None, "error": str(e)}

def score_dataset(texts, concepts_df, max_workers=4, checkpoint_path="score_ckpt.jsonl"):
    rows, done_ids = [], set()
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            for line in f:
                rec = json.loads(line)
                rows.append(rec)
                done_ids.add(rec["orig_index"])

    items = [(i, t) for i, t in enumerate(texts) if i not in done_ids]
    print(f"Already scored: {len(done_ids)}, Remaining: {len(items)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_score_one, idx, txt, concepts_df): idx for idx, txt in items}
        with open(checkpoint_path, "a") as fout:
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Scoring"):
                result = fut.result()
                fout.write(json.dumps(result) + "\n")
                fout.flush()
                rows.append(result)
    return pd.DataFrame(rows)

In [ ]:
fn_scores_raw = score_dataset(
    fn_texts, fn_concepts, max_workers=4,
    checkpoint_path=f"{OUTPUT_DIR}/fn_score_checkpoint.jsonl"
)
fp_scores_raw = score_dataset(
    fp_texts, fp_concepts, max_workers=4,
    checkpoint_path=f"{OUTPUT_DIR}/fp_score_checkpoint.jsonl"
)

def expand_scored_rows(scored_df):
    rows = []
    for _, row in scored_df.iterrows():
        matches = row.get("matches")
        if isinstance(matches, list):
            for m in matches:
                rows.append({"orig_index": row["orig_index"], "text": row["text"],
                             "concept_name": m.get("concept_name"), "match": m.get("match"),
                             "reason": m.get("reason"), "error": row.get("error")})
        else:
            rows.append({"orig_index": row["orig_index"], "text": row["text"],
                         "concept_name": None, "match": None, "reason": None, "error": row.get("error")})
    return pd.DataFrame(rows)

fn_scores = expand_scored_rows(fn_scores_raw)
fp_scores = expand_scored_rows(fp_scores_raw)

fn_scores.to_csv(f"{OUTPUT_DIR}/fn_scores.csv", index=False)
fp_scores.to_csv(f"{OUTPUT_DIR}/fp_scores.csv", index=False)

print("FN scores:", fn_scores.shape)
print("FP scores:", fp_scores.shape)

# Step 5: Visualizations

In [ ]:
def summarize_concept_coverage(scores_df):
    matched = scores_df[scores_df["match"] == 1].copy()
    by_concept = matched["concept_name"].value_counts().reset_index()
    by_concept.columns = ["concept_name", "matched_count"]
    return by_concept

fn_summary = summarize_concept_coverage(fn_scores)
fp_summary = summarize_concept_coverage(fp_scores)

fn_summary.to_csv(f"{OUTPUT_DIR}/fn_concept_summary.csv", index=False)
fp_summary.to_csv(f"{OUTPUT_DIR}/fp_concept_summary.csv", index=False)

display(fn_summary)
display(fp_summary)

In [ ]:
def wrap_labels(labels, width=40):
    return ["\n".join(textwrap.wrap(str(l), width=width)) for l in labels]

def concept_barplot(df, title, top_n=10, save_path=None):
    plot_df = df.sort_values("matched_count", ascending=False).head(top_n).iloc[::-1]
    wrapped = wrap_labels(plot_df["concept_name"], width=45)
    plt.figure(figsize=(12, 7))
    plt.barh(wrapped, plot_df["matched_count"])
    plt.yticks(fontweight="bold")
    plt.xlabel("Matched Count", fontweight="bold")
    plt.ylabel("Concept", fontweight="bold")
    plt.title(title, fontweight="bold")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

concept_barplot(fn_summary, "Inciting FN — Top Concepts (Inciting missed as None)",
                save_path=f"{OUTPUT_DIR}/fn_concepts_barplot.pdf")
concept_barplot(fp_summary, "Inciting FP — Top Concepts (None flagged as Inciting)",
                save_path=f"{OUTPUT_DIR}/fp_concepts_barplot.pdf")

In [ ]:
def concept_treemap(df, title, top_n=10):
    plot_df = df.sort_values("matched_count", ascending=False).head(top_n)
    labels = ["\n".join(textwrap.wrap(str(n), width=20)) + f"\n({c})"
              for n, c in zip(plot_df["concept_name"], plot_df["matched_count"])]
    plt.figure(figsize=(14, 8))
    squarify.plot(sizes=plot_df["matched_count"], label=labels, alpha=0.8)
    plt.axis("off")
    plt.title(title, fontsize=16)
    plt.show()

concept_treemap(fn_summary, "Inciting FN — Concept Treemap")
concept_treemap(fp_summary, "Inciting FP — Concept Treemap")

In [ ]:
def concept_cooccurrence_heatmap(scores_df, title, top_n=10):
    matched = scores_df[scores_df["match"] == 1].copy()
    top_concepts = matched["concept_name"].value_counts().head(top_n).index.tolist()
    matched = matched[matched["concept_name"].isin(top_concepts)]
    binary = pd.crosstab(matched["orig_index"], matched["concept_name"])
    cooc = binary.T.dot(binary)
    plt.figure(figsize=(10, 8))
    plt.imshow(cooc, interpolation="nearest")
    plt.colorbar(label="Shared matched texts")
    plt.xticks(range(len(cooc.columns)), ["\n".join(textwrap.wrap(c, 20)) for c in cooc.columns], rotation=90)
    plt.yticks(range(len(cooc.index)), ["\n".join(textwrap.wrap(c, 20)) for c in cooc.index])
    plt.title(title)
    plt.tight_layout()
    plt.show()

concept_cooccurrence_heatmap(fn_scores, "Inciting FN — Concept Co-occurrence")
concept_cooccurrence_heatmap(fp_scores, "Inciting FP — Concept Co-occurrence")

In [ ]:
print("All outputs saved to:", OUTPUT_DIR)
print("Files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")